In [2]:
# ==============================================================================
# 1. Import Necessary Libraries
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

In [3]:
# Set plotting style for clear, professional visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [17]:
# ==============================================================================
# 2. Download and Load the Dataset via Kagglehub
# ==============================================================================
print("Downloading dataset from Kaggle...")
# Download the latest version of the dataset
dataset_path = kagglehub.dataset_download("pratyushpuri/financial-news-market-events-dataset-2025")

print(f"Dataset downloaded to cache: {dataset_path}")

# Dynamically find the CSV file in the downloaded directory
csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]

if not csv_files:
    raise FileNotFoundError("No CSV file found in the downloaded Kaggle dataset.")

# Assuming the main data is in the first CSV found
file_path = os.path.join(dataset_path, csv_files[0])
df = pd.read_csv(file_path)

# Quick sanity check on shape
print(f"\nDataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

Dataset downloaded to cache: /Users/tajsharma/.cache/kagglehub/datasets/pratyushpuri/financial-news-market-events-dataset-2025/versions/1

Dataset Shape: 3024 rows, 12 columns



In [18]:
# ==============================================================================
# 3. Initial Data Quality & Null Inspection
# ==============================================================================
print("--- Percentage of Missing Values per Column ---")
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage[missing_percentage > 0].sort_values(ascending=False))

--- Percentage of Missing Values per Column ---
Sentiment               5.654762
Index_Change_Percent    5.324074
News_Url                5.059524
Headline                4.894180
dtype: float64


In [19]:
# ==============================================================================
# 4. Systematically Handling the 5% Null Rate
# ==============================================================================
df_cleaned = df.copy()

# Drop rows where the actual 'Headline' is missing (required for NLP/Context)
if 'Headline' in df_cleaned.columns:
    df_cleaned.dropna(subset=['Headline'], inplace=True)

# Separate categorical and numerical columns for systematic imputation
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns
numerical_cols = df_cleaned.select_dtypes(include=['number']).columns

# Impute Categorical Columns with 'Unknown'
for col in categorical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna('Unknown')

# Impute Numerical Columns with Median
for col in numerical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# Final Verification
print("\n--- Missing Values Post-Cleaning ---")
print(df_cleaned.isnull().sum())
print(f"\nFinal Cleaned Dataset Shape: {df_cleaned.shape[0]} rows, {df_cleaned.shape[1]} columns")


--- Missing Values Post-Cleaning ---
Date                    0
Headline                0
Source                  0
Market_Event            0
Market_Index            0
Index_Change_Percent    0
Trading_Volume          0
Sentiment               0
Sector                  0
Impact_Level            0
Related_Company         0
News_Url                0
dtype: int64

Final Cleaned Dataset Shape: 2876 rows, 12 columns


/var/folders/k5/1ys4jvv95zv528gdmtsrb7s80000gn/T/ipykernel_29633/472053604.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_cleaned.select_dtypes(include=['object']).columns
